In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import joblib
import ta
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

In [2]:
# Configurações Base

time_steps = 7

tickers = {
    'Bovespa': '^BVSP',        
    'Dolar': 'BRL=X',        
    'SP500': '^GSPC',        
    'Shanghai': '000001.SS', # Bolsa da China
    'Petroleo': 'BZ=F',      
    'Minerio': 'TIO=F',      
    'Ouro': 'GC=F',
    'Juros_BR': 'LFTS11.SA',
    'Inflacao_BR': 'IMAB11.SA'
}

nomes_classes = {
    0: "📉 BAIXA FORTE (menor que -1.0%)",
    1: "↘️ LEVE BAIXA (-1.0% a -0.2%)",
    2: "➖ NEUTRO (-0.2% a +0.2%)",
    3: "↗️ LEVE ALTA (+0.2% a +1.0%)",
    4: "📈 ALTA FORTE (maior que +1.0%)"
}

print("📥 Baixando cotações atualizadas...")
df_precos = yf.download(list(tickers.values()), period='2mo')['Close']
df_precos.rename(columns={v: k for k, v in tickers.items()}, inplace=True)
df_precos.ffill(inplace=True)
df_precos.dropna(inplace=True)

📥 Baixando cotações atualizadas...


[*********************100%***********************]  9 of 9 completed


In [3]:
df_features = pd.DataFrame(index=df_precos.index)
for col in df_precos.columns:
    df_features[col] = df_precos[col].pct_change()

df_features['RSI'] = ta.momentum.RSIIndicator(df_precos['Bovespa'], window=14).rsi()
sma_15 = ta.trend.SMAIndicator(df_precos['Bovespa'], window=15).sma_indicator()
df_features['Distancia_SMA15'] = (df_precos['Bovespa'] / sma_15) - 1
df_features['Bollinger_Width'] = ta.volatility.BollingerBands(df_precos['Bovespa'], window=20).bollinger_wband()

df_features.dropna(inplace=True)
colunas_features = df_features.columns
ultimos_dias = df_features.tail(time_steps).values

In [4]:
caminho_modelos = os.path.join(os.getcwd(), 'modelos')

In [5]:
# Carrega o normalizador do modelo de 5 classes
scaler_X = joblib.load(os.path.join(caminho_modelos, 'scaler_X_classificador.pkl'))
X_futuro_scaled = scaler_X.transform(ultimos_dias.reshape(-1, ultimos_dias.shape[-1])).reshape(1, time_steps, len(colunas_features))

In [6]:
# Reconstrói a rede com 5 saídas
modelo_gru = Sequential([
    GRU(50, return_sequences=True, input_shape=(time_steps, len(colunas_features))),
    Dropout(0.2),
    GRU(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(5, activation='softmax') # 5 neurónios
])

C:\Users\flavi\anaconda3\envs\SeriesTemporais\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [7]:
# Carrega os pesos de 5 classes
modelo_gru.load_weights(os.path.join(caminho_modelos, 'modelo_gru_classificador.weights.h5'))

In [8]:
probabilidades = modelo_gru.predict(X_futuro_scaled, verbose=0)[0]
classe_vencedora = np.argmax(probabilidades)

fechamento_hoje = df_precos['Bovespa'].iloc[-1]
lim_baixa_forte = fechamento_hoje * (1 - 0.01)   # -1%
lim_leve_baixa  = fechamento_hoje * (1 - 0.002)  # -0.2%
lim_leve_alta   = fechamento_hoje * (1 + 0.002)  # +0.2%
lim_alta_forte  = fechamento_hoje * (1 + 0.01)   # +1%

risco_volatilidade = (probabilidades[4] + probabilidades[0]) * 100

In [9]:
print("\n" + "="*55)
print(" 🚀 RADAR BOVESPA: 5 CLASSES (RISCO & VOLATILIDADE) 🚀")
print("="*55)
print(f"Cotação Atual Base: {fechamento_hoje:.0f} pts (Data: {df_precos.index[-1].strftime('%d/%m/%Y')})")
print(f"Veredito Principal: {nomes_classes[classe_vencedora]}")

print("\n📊 RAIO-X DE PROBABILIDADES:")
print(f"   🟩 Alta Forte:  {probabilidades[4]*100:>5.2f}%")
print(f"   🟢 Leve Alta:   {probabilidades[3]*100:>5.2f}%")
print(f"   🟡 Neutro:      {probabilidades[2]*100:>5.2f}%")
print(f"   🟠 Leve Baixa:  {probabilidades[1]*100:>5.2f}%")
print(f"   🟥 Baixa Forte: {probabilidades[0]*100:>5.2f}%")
print("-" * 55)
print(f"   ⚠️ Risco de Movimento Violento (Extremos): {risco_volatilidade:.2f}%")

print("\n🎯 ZONAS DE PREÇO (PRÓXIMO DIA ÚTIL):")
print(f"   Acima de {lim_alta_forte:.0f} pts         ➡️ ALTA FORTE")
print(f"   Entre {lim_leve_alta:.0f} e {lim_alta_forte:.0f} pts ➡️ LEVE ALTA")
print(f"   Entre {lim_leve_baixa:.0f} e {lim_leve_alta:.0f} pts ➡️ NEUTRO")
print(f"   Entre {lim_baixa_forte:.0f} e {lim_leve_baixa:.0f} pts ➡️ LEVE BAIXA")
print(f"   Abaixo de {lim_baixa_forte:.0f} pts        ➡️ BAIXA FORTE")
print("="*55)


 🚀 RADAR BOVESPA: 5 CLASSES (RISCO & VOLATILIDADE) 🚀
Cotação Atual Base: 175135 pts (Data: 28/08/2026)
Veredito Principal: ➖ NEUTRO (-0.2% a +0.2%)

📊 RAIO-X DE PROBABILIDADES:
   🟩 Alta Forte:   9.54%
   🟢 Leve Alta:   27.16%
   🟡 Neutro:      31.59%
   🟠 Leve Baixa:  16.12%
   🟥 Baixa Forte: 15.59%
-------------------------------------------------------
   ⚠️ Risco de Movimento Violento (Extremos): 25.12%

🎯 ZONAS DE PREÇO (PRÓXIMO DIA ÚTIL):
   Acima de 176887 pts         ➡️ ALTA FORTE
   Entre 175486 e 176887 pts ➡️ LEVE ALTA
   Entre 174785 e 175486 pts ➡️ NEUTRO
   Entre 173384 e 174785 pts ➡️ LEVE BAIXA
   Abaixo de 173384 pts        ➡️ BAIXA FORTE
